# Hypergraph

From the corpus to the hypergraph of the report's Section 3.1. Reads `data/interim/01_corpus.csv` and the raw scores; writes `tex/figures/hyperedge_shapes.pdf`.

- **Draft:** only the figure runs. The sections after it state what they will compute; their code is the next step.

In [ ]:
import shutil, subprocess, tempfile
from pathlib import Path
import numpy as np
import pandas as pd
from matplotlib.colors import to_rgb
from music21 import converter
from IPython.display import Image, display
from figures import palette   # the Nord colours of the house style

RAW = Path("../data/raw")
INTERIM = Path("../data/interim")
FIGURES = Path("../tex/figures")   # read by the report
corpus = pd.read_csv(INTERIM / "01_corpus.csv").set_index("file")

## Hyperedge shapes on the score

- **Why:** Figure 1 of the report, a hyperedge of shape v × s drawn as a coloured blob around the notes of v voices over s cells.
- **Excerpt:** the first two bars of BWV 40.6, transposed as the model reads them: four voices in quarter notes, then a half note; one accidental, the bass's F♯. Not one of the three Bach files under the CCARH copyright.
- **Four hyperedges, one of each shape, in Nord colours:** 4 × 2 on beats 1–2, 2 × 2 (soprano, alto) on beats 3–4, then in bar 2 2 × 1 (tenor, bass) on beat 1 and 1 × 2 (soprano) on beats 2–3; no two overlap. A legend on the right; one column of the report wide.
- **Drawing:** MusiXTeX engraves the bars on four staves; TikZ draws each hyperedge behind the notes, at positions that `tikzmark` saves on the noteheads, so LaTeX runs twice.
- **Simplification:** one cell per beat for readability; the model's cell is a quarter of a beat.

In [ ]:
FILE, BARS = "bwv40.6.mxl", 2
row = corpus.loc[FILE]
score = converter.parse(RAW / "bach" / FILE)
parts = [int(k) for k in row["voice_parts"].split("|")]
top = score.parts[parts[0]]
start = float(top.measure(1).getOffsetBySite(top))
notes = [[n for n in score.parts[k].stripTies().flatten().notes if start <= float(n.offset) < start + 4 * BARS] for k in parts]
assert all([float(n.quarterLength) for n in v] == [1, 1, 1, 1, 1, 1, 2] for v in notes)   # six quarter notes, then a half note
HALF = 6   # the index of the half note, on beats 3-4 of bar 2
pitches = [[n.pitch.midi + int(row["shift"]) for n in v] for v in notes]   # [voice][note], transposed
pitches

In [ ]:
STEP = {0: (0, ""), 1: (0, "sh"), 2: (1, ""), 3: (1, "sh"), 4: (2, ""), 5: (3, ""), 6: (3, "sh"),
        7: (4, ""), 8: (4, "sh"), 9: (5, ""), 10: (5, "sh"), 11: (6, "")}   # pitch class to diatonic step and accidental, black keys as sharps
BOTTOM = [30, 30, 23, 18]   # diatonic number of each staff's bottom line: E4 for soprano and alto, E3 for the tenor's octave treble, G2 for the bass

def position(v, midi):   # MusiXTeX's note position, 0 on the bottom line, and the accidental
    step, accidental = STEP[midi % 12]
    return (midi // 12 - 1) * 7 + step - BOTTOM[v], accidental

nord = palette["categorical"]
HYPEREDGES = [("1 \\texttimes{} 2", "one voice, two cells", nord[5], [(0, 5), (0, 6)]),   # cells: (voice, note index)
              ("2 \\texttimes{} 1", "two voices, one cell", nord[4], [(2, 4), (3, 4)]),
              ("2 \\texttimes{} 2", "two voices, two cells", nord[1], [(0, 2), (1, 2), (0, 3), (1, 3)]),
              ("4 \\texttimes{} 2", "four voices, two cells", nord[0], [(v, b) for v in range(4) for b in (0, 1)])]

def excerpt():   # the two bars on four staves, a tikzmark at the centre of every notehead
    lines = [r"{\smallmusicsize\elemskip=2.6pt", r"\begin{music}", r"\parindent0mm\nobarnumbers\interstaff{9}", r"\instrumentnumber{4}",
             r"\setstaffs11\setclef1{6}", r"\setstaffs21\setclef2{0}\setclefsymbol2{\treblelowoct}", r"\setstaffs31\setclef3{0}", r"\setstaffs41\setclef4{0}",
             r"\generalsignature{0}\generalmeter{\meterfrac44}", r"\startextract"]
    n = len(pitches[0])
    for b in range(n):
        fields = []
        for v in (3, 2, 1, 0):   # MusiXTeX counts instruments from the bottom staff
            p, accidental = position(v, pitches[v][b])
            mark = r"\zcharnote{%d}{\kern0.6\Interligne\tikzmark{p-%d-%d}}" % (p, v, b)
            note = ("h" if b == HALF else "q") + ("u" if p < 4 else "l")   # stem up below the middle line
            fields.append(mark + (r"\%s{%d}" % (accidental, p) if accidental else "") + r"\%s{%d}" % (note, p))
        wide = b == HALF or (b + 1 < n and any(position(v, pitches[v][b + 1])[1] for v in range(4)))   # room for a half note, or before an accidental
        lines.append((r"\NOTes " if wide else r"\NOtes ") + "&".join(fields) + r"\en" + (r"\bar" if b == 3 else ""))
    return "\n".join(lines + [r"\zendextract", r"\end{music}}"]) + "\n"

def hull(k, cells):   # the window's corners, clockwise from the top left, filled and stroked 9 pt wide so the blob reaches past the noteheads
    voices, beats = sorted({v for v, _ in cells}), sorted({b for _, b in cells})
    corners = [(voices[0], beats[0]), (voices[0], beats[-1]), (voices[-1], beats[-1]), (voices[-1], beats[0])]
    path = " -- ".join(f"(pic cs:p-{v}-{b})" for v, b in dict.fromkeys(corners))
    return rf"\fill[h{k}] {path} -- cycle; \draw[h{k}, line width=9pt, line join=round, line cap=round] {path} -- cycle;"

shades = [r"\definecolor{h%d}{rgb}{%.4f,%.4f,%.4f}" % (k, *(0.45 * np.array(to_rgb(c)) + 0.55)) for k, (_, _, c, _) in enumerate(HYPEREDGES)]   # lightened on white, opaque
legend = r"\\[4pt]".join(rf"\tikz[baseline=0.4ex]\fill[h{k}, rounded corners=2pt] (0,0) rectangle (0.45,0.3); & {shape}: {words}" for k, (shape, words, _, _) in enumerate(HYPEREDGES))
doc = [r"\documentclass[border=2pt]{standalone}", r"\usepackage{musixtex}", r"\usepackage{tikz}", r"\usetikzlibrary{tikzmark}", r"\usepackage{array}", *shades,
       r"\begin{document}",
       r"\tikz[remember picture,overlay]{" + " ".join(hull(k, cells) for k, (_, _, _, cells) in enumerate(HYPEREDGES)) + "}%",   # before the music, so behind the notes
       r"\parbox[c]{4.9cm}{\input{excerpt.tex}}\hspace{8pt}%",   # MusiXTeX needs its own file: inside an argument, & would be read as a column
       r"\begin{tabular}[c]{@{}c@{\hspace{4pt}}>{\sffamily\footnotesize\raggedright\arraybackslash}p{2.45cm}@{}}" + legend + r"\end{tabular}",
       r"\end{document}"]

with tempfile.TemporaryDirectory() as tmp:
    Path(tmp, "excerpt.tex").write_text(excerpt())
    Path(tmp, "shapes.tex").write_text("\n".join(doc) + "\n")
    run = subprocess.run(["latexmk", "-lualatex", "-interaction=nonstopmode", "shapes.tex"], cwd=tmp, capture_output=True, text=True)
    assert run.returncode == 0, run.stdout[-2000:]   # fail loud, with LaTeX's own message
    FIGURES.mkdir(parents=True, exist_ok=True)
    shutil.copy(Path(tmp, "shapes.pdf"), FIGURES / "hyperedge_shapes.pdf")
    subprocess.run(["pdftoppm", "-png", "-r", "150", "-singlefile", "shapes.pdf", "preview"], cwd=tmp, check=True)   # a PNG to show here: Jupyter does not draw a PDF
    display(Image(filename=str(Path(tmp, "preview.png"))))
print("Written:", FIGURES / "hyperedge_shapes.pdf")

## Grid (to build)

- **Rule:** each voice of each row of `02_corpus` on a grid of a quarter of the beat; each cell keeps the note sounding at its end (01_filtering); pitches moved by the file's shift; the skipped spans of 02_corpus left out.
- **Value:** a pitch or a rest, and an attack flag: the voice attacks a new note in the cell, or holds the note of the cell before.
- **Checks:** the off-grid notes are the nine Bach ornaments of 01_filtering; a toy score computed by hand.

## Windows and classes (to build)

- **Shapes:** every v × s with v and s from 1 to 4; in Palestrina, every set of v voices of the movement.
- **Class:** the window's values in two representations: the exact pitches, and the intervals from the lowest note of its first cell.
- **Rule:** no window spans a skipped span.
- **Checks:** the number of windows per file, from the size of its grid.

## Support (to build)

- **Rule:** the support of a class is the share of the windows of its shape that belong to it, each window counted 1/k times for a file with k versions (Agrawal et al. 1993 for the term).
- **Per fold:** the support on the complete corpus and on each set of nine folds.
- **Checks:** the supports of a shape sum to 1; the ten folds sum to the complete corpus.

## Coverage (to build)

- **Measure:** per shape and corpus, the number of classes, the share of classes seen once, and the coverage: the share of the windows of a held-out fold whose class has a positive support in the other nine folds.
- **On two sets of windows:** all windows, and the windows where at least one voice attacks a new note.
- **Threshold:** fixed before the measures.